# Шаг 5 (ключевой): PPML-проверка робастности — HS 284690

**Это ключевой ноутбук результатов.** Проверяет основной вывод с помощью PPML (Poisson Pseudo Maximum Likelihood), который корректно обрабатывает нулевые торговые потоки и гетероскедастичность (Santos Silva & Tenreyro, 2006).

**Исследовательский вопрос:** Устойчив ли наблюдаемый эффект к выбору оценщика?

**Метод:** PPML с фиксированными эффектами партнёра и года. Переменная воздействия: `Post2016 × HighExposure`. SE кластеризованы по партнёру.

**Основной результат:** Высокозависимые страны значимо снизили импорт после реформы. Эффект устойчив по стоимости и физическому объёму, по альтернативным порогам (топ-10/топ-20) и наборам контролей.

**Входные данные:** `../data/china_ree_exports_with_controls.xlsx`  
**Выходные данные:** `../results/ppml_robustness_results_hs284690.xlsx`

## 0. Импорты и конфигурация

In [1]:
import math
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

INPUT_FILE  = "../data/china_ree_exports_with_controls.xlsx"
OUTPUT_FILE = "../results/ppml_robustness_results_hs284690.xlsx"

SPECIAL_PARTNERS_TO_DROP = ["Other Asia, nes"]

## 1. Построение групп экспозиции

`add_exposures` ранжирует страны-партнёры по базовому периоду 2010–2014 и присваивает бинарные индикаторы топ-10/топ-15/топ-20.

In [2]:
def add_exposures(df, metric, prefix):
    """Строит группы высокой экспозиции по среднему значению метрики за 2010–2014."""
    base = df[df["year"].between(2010, 2014)].copy()
    avg = base.groupby("partner")[metric].mean().sort_values(ascending=False)
    for n in [10, 15, 20]:
        top = avg.head(n).index.tolist()
        col = f"{prefix}_top{n}"
        df[col] = df["partner"].isin(top).astype(int)
        df[f"post2016_{col}"] = df["post2016"] * df[col]
        df[f"post2015_{col}"] = df["post2015"] * df[col]
    return df, avg

## 2. PPML-оценка

`ppml_fit` оценивает модель PPML (GLM семейства Пуассона) с фиксированными эффектами партнёра и года.
Использование PPML вместо OLS на ln(1+x) обосновано Santos Silva & Tenreyro (2006): модель корректно обрабатывает нулевые торговые потоки и гетероскедастичность.
Стандартные ошибки кластеризованы по стране-партнёру.

In [3]:
def ppml_fit(data, depvar, treatment, controls=None, fe_spec="two_way_fe", cluster="partner"):
    """PPML с фиксированными эффектами партнёра и года. Возвращает словарь результатов с exp(beta)-1 эффектом."""
    controls = controls or []
    reg = data.copy()
    needed = [depvar, treatment, cluster, "partner", "year"] + controls
    reg = reg.dropna(subset=needed)
    reg = reg[reg[depvar] >= 0].copy()

    terms = [treatment] + controls
    if fe_spec in ["country_fe", "two_way_fe"]:
        terms.append("C(partner)")
    if fe_spec in ["year_fe", "two_way_fe"]:
        terms.append("C(year)")

    formula = depvar + " ~ " + " + ".join(terms)

    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model = smf.glm(formula=formula, data=reg, family=sm.families.Poisson())
            result = model.fit(
                maxiter=200,
                cov_type="cluster",
                cov_kwds={"groups": reg[cluster]},
            )
            warning_text = "; ".join(str(w.message) for w in caught[:5])

        coef = result.params.get(treatment, float("nan"))
        se   = result.bse.get(treatment, float("nan"))
        pval = result.pvalues.get(treatment, float("nan"))
        fitted = result.predict(reg)
        rmse = float(np.sqrt(np.mean((reg[depvar] - fitted) ** 2)))

        try:
            # Псевдо-R2 Макфаддена
            pseudo_r2 = 1 - result.deviance / result.null_deviance
        except Exception:
            pseudo_r2 = float("nan")

        return {
            "depvar": depvar,
            "treatment": treatment,
            "controls": "+".join(controls) if controls else "none",
            "fe_spec": fe_spec,
            "nobs": int(result.nobs),
            "coef_treatment": float(coef),
            "se_cluster_partner": float(se),
            "z_treatment": float(coef / se) if se and se == se else float("nan"),
            "p_cluster_partner": float(pval),
            # exp(beta)-1: мультипликативный эффект на условное среднее
            "effect_percent_exp_beta_minus1": float((math.exp(coef) - 1) * 100),
            "pseudo_r2_deviance": float(pseudo_r2),
            "rmse_level": rmse,
            "aic": float(result.aic),
            "converged": bool(result.converged),
            "status": "ok",
            "warnings": warning_text,
            "error": "",
        }

    except Exception as e:
        return {
            "depvar": depvar, "treatment": treatment,
            "controls": "+".join(controls) if controls else "none",
            "fe_spec": fe_spec, "nobs": len(reg),
            "coef_treatment": float("nan"), "se_cluster_partner": float("nan"),
            "z_treatment": float("nan"), "p_cluster_partner": float("nan"),
            "effect_percent_exp_beta_minus1": float("nan"),
            "pseudo_r2_deviance": float("nan"), "rmse_level": float("nan"),
            "aic": float("nan"), "converged": False,
            "status": "error", "warnings": "", "error": repr(e),
        }

## 3. Спецификации и запуск

Набор оцениваемых спецификаций:
- **основная спецификация**: стоимость и физический объём, топ-15, без контролей
- **робастность**: альтернативные пороги (топ-10/топ-20), год отсечения 2015, физический объём и доля как метрика экспозиции
- **контроли**: с ВВП; с ВВП + доля обрабатывающей промышленности
- **положительные потоки**: только ненулевые торговые потоки
- **дополнительные шоки**: с взаимодействиями торговой войны и COVID

In [4]:
def main():
    df = pd.read_excel(INPUT_FILE, sheet_name="panel_with_controls")

    numeric_cols = ["year", "export_value_1000usd", "export_value_usd",
                    "quantity_kg", "ln_gdp", "manufacturing_share_gdp",
                    "share_export_value", "post2015", "post2016", "trade_war", "covid"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df[~df["partner"].isin(SPECIAL_PARTNERS_TO_DROP)].copy()

    df, avg_value    = add_exposures(df, "export_value_usd",  "high_value")
    df, avg_quantity = add_exposures(df, "quantity_kg",        "high_quantity")
    df, avg_share    = add_exposures(df, "share_export_value", "high_share")

    df["trade_war_us"]          = df["trade_war"] * df["partner"].eq("United States").astype(int)
    df["covid_high_value_top15"] = df["covid"] * df["high_value_top15"]

    top_exposure = (
        avg_value.reset_index()
        .rename(columns={"export_value_usd": "baseline_avg_export_value_usd_2010_2014"})
    )
    top_exposure.insert(0, "rank", range(1, len(top_exposure) + 1))
    top_exposure["high_value_top10"] = (top_exposure["rank"] <= 10).astype(int)
    top_exposure["high_value_top15"] = (top_exposure["rank"] <= 15).astype(int)
    top_exposure["high_value_top20"] = (top_exposure["rank"] <= 20).astype(int)

    specs = []

    # Основная спецификация: два типа зависимой переменной
    for depvar in ["export_value_1000usd", "quantity_kg"]:
        specs.append({
            "suite": "final_ppml_main", "depvar": depvar,
            "treatment": "post2016_high_value_top15",
            "controls": [], "fe_spec": "two_way_fe", "positive_only": False,
        })

    # Альтернативные группы экспозиции и год отсечения
    for treatment in ["post2015_high_value_top15", "post2016_high_value_top10",
                      "post2016_high_value_top20", "post2016_high_quantity_top15",
                      "post2016_high_share_top15"]:
        specs.append({
            "suite": "ppml_robustness_value", "depvar": "export_value_1000usd",
            "treatment": treatment, "controls": [], "fe_spec": "two_way_fe",
            "positive_only": False,
        })

    # Контрольные переменные
    for controls in [["ln_gdp"], ["ln_gdp", "manufacturing_share_gdp"]]:
        specs.append({
            "suite": "ppml_controls_value", "depvar": "export_value_1000usd",
            "treatment": "post2016_high_value_top15",
            "controls": controls, "fe_spec": "two_way_fe", "positive_only": False,
        })

    # Только ненулевые потоки
    specs.append({
        "suite": "ppml_positive_flows_value", "depvar": "export_value_1000usd",
        "treatment": "post2016_high_value_top15",
        "controls": [], "fe_spec": "two_way_fe", "positive_only": True,
    })

    # Шоки торговой войны и COVID
    specs.append({
        "suite": "ppml_extra_shocks_value", "depvar": "export_value_1000usd",
        "treatment": "post2016_high_value_top15",
        "controls": ["trade_war_us", "covid_high_value_top15"],
        "fe_spec": "two_way_fe", "positive_only": False,
    })

    rows = []
    for spec in specs:
        reg_data = df[df[spec["depvar"]] > 0].copy() if spec["positive_only"] else df.copy()
        row = ppml_fit(
            data=reg_data, depvar=spec["depvar"], treatment=spec["treatment"],
            controls=spec["controls"], fe_spec=spec["fe_spec"],
        )
        row["suite"] = spec["suite"]
        row["positive_only"] = spec["positive_only"]
        rows.append(row)

    results = pd.DataFrame(rows)

    readme = pd.DataFrame({
        "item": ["method", "основная зависимая переменная", "основная переменная воздействия",
                 "фиксированные эффекты", "стандартные ошибки", "исключенная категория", "interpretation"],
        "value": [
            "PPML-проверка робастности", "export_value_1000usd",
            "post2016_high_value_top15", "фиксированные эффекты партнёра и года",
            "кластеризация по партнеру", "Other Asia, nes",
            "exp(beta)-1 дает мультипликативное различие условного среднего",
        ],
    })

    methodology_text = pd.DataFrame({
        "section": ["Методологическое обоснование", "Спецификация",
                    "Интерпретация", "Ограничение"],
        "text": [
            "PPML используется как дополнительная проверка устойчивости: торговые данные содержат нулевые потоки и гетероскедастичность.",
            "ExportValue_it = exp(alpha_i + lambda_t + beta * Post2016_t x HighExposure_i) + error_it.",
            "Отрицательный и значимый коэффициент означает более низкое условное среднее значение экспорта для стран группы высокой экспозиции после 2016 г. относительно остальных.",
            "PPML не превращает модель в строгий каузальная DID-оценка; результаты остаются эмпирической структурной ассоциацией.",
        ],
    })

    sources = pd.DataFrame({
        "source": ["Santos Silva & Tenreyro (2006), The Log of Gravity", "WITS / UN Comtrade"],
        "url": ["https://ideas.repec.org/a/tpr/restat/v88y2006i4p641-658.html",
                "https://wits.worldbank.org/"],
    })

    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        readme.to_excel(writer, sheet_name="README", index=False)
        results[results["suite"] == "final_ppml_main"].to_excel(
            writer, sheet_name="ppml_main", index=False)
        results[results["suite"] != "final_ppml_main"].to_excel(
            writer, sheet_name="ppml_robustness", index=False)
        top_exposure.head(25).to_excel(writer, sheet_name="top_exposure_clean", index=False)
        methodology_text.to_excel(writer, sheet_name="methodology_text", index=False)
        sources.to_excel(writer, sheet_name="sources", index=False)

    print("Готово.")
    print(f"Результаты сохранены в: {OUTPUT_FILE}")
    print()
    print("Основные результаты PPML:")
    print(results[results["suite"] == "final_ppml_main"][[
        "depvar", "coef_treatment", "se_cluster_partner",
        "p_cluster_partner", "effect_percent_exp_beta_minus1", "nobs", "status",
    ]].to_string(index=False))

## 4. Запуск

In [5]:
main()

Готово.
Результаты сохранены в: ../results/ppml_robustness_results_hs284690.xlsx

Основные результаты PPML:
              depvar  coef_treatment  se_cluster_partner  p_cluster_partner  effect_percent_exp_beta_minus1  nobs status
export_value_1000usd       -1.366836            0.309362           0.000010                      -74.508785  1185     ok
         quantity_kg       -1.180785            0.269964           0.000012                      -69.296248  1185     ok


## Результаты PPML

### Основная спецификация (HS 284690, двусторонние фиксированные эффекты, кластерные SE по партнёру, N = 1 185)

| Зависимая переменная | β | SE | p | exp(β)−1 |
|---|---|---|---|---|
| Стоимость экспорта (тыс. долл.) | **−1.367** | 0.309 | < 0.001 *** | −74.5% |
| Физический объём (кг) | **−1.181** | 0.270 | < 0.001 *** | −69.3% |

**Интерпретация:** Условное среднее значение экспорта для группы высокой экспозиции
после 2016 года ниже примерно на 70–75% относительно группы низкой экспозиции
при прочих равных (фиксированные эффекты партнёра и года).

**Значение PPML-проверки:** OLS на ln(1+x) может давать смещённые оценки при наличии
нулевых потоков и гетероскедастичности. Согласованность PPML и OLS-результатов
подтверждает устойчивость основного вывода к выбору оценщика.

### Устойчивость (стоимость экспорта, все спецификации значимы p < 0.05)

Результат воспроизводится при:
- альтернативном годе отсечения (2015)
- порогах топ-10 и топ-20
- экспозиции по физическому объёму и доле
- добавлении контролей ВВП и доли обрабатывающей промышленности
- ограничении на ненулевые потоки
- включении взаимодействий торговой войны и COVID

**Ограничение:** PPML не устраняет проблему нарушенных pre-trends (p = 0.0038 по F-тесту).
Результаты остаются эмпирической структурной ассоциацией, не строгой каузальной DID-оценкой.
